In [7]:
# ============================================================
# FINAL HAYNES 282 ML MODEL TRAINING
# FOUR PROPERTIES + DEFECT CLASSIFICATION
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    cross_val_predict
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    PolynomialFeatures
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    f1_score,
    confusion_matrix
)

from catboost import (
    CatBoostRegressor,
    CatBoostClassifier
)


# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv(
    "haynes282_final_dataset.csv"
)

print("=" * 70)
print("DATASET")
print("=" * 70)

print("Shape:", df.shape)


# ============================================================
# 2. FEATURES
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

X = df[features]

joblib.dump(
    features,
    "feature_order.pkl"
)


# ============================================================
# 3. CV
# ============================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# 4. FUNCTION TO CREATE FRESH REGRESSION MODEL
# ============================================================

def make_regression_model(name):

    if name == "Linear Regression":

        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ])


    elif name == "Ridge":

        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10))
        ])


    elif name == "Lasso":

        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=1))
        ])


    elif name == "Polynomial Degree 2 + Ridge":

        return Pipeline([
            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False
                )
            ),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10))
        ])


    elif name == "Polynomial Degree 3 + Ridge":

        return Pipeline([
            (
                "poly",
                PolynomialFeatures(
                    degree=3,
                    include_bias=False
                )
            ),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10))
        ])


    elif name == "Extra Trees":

        return ExtraTreesRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        )


    elif name == "Random Forest":

        return RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        )


    elif name == "Gradient Boosting":

        return GradientBoostingRegressor(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.03,
            loss="huber",
            random_state=42
        )


    elif name == "CatBoost":

        return CatBoostRegressor(
            iterations=300,
            depth=5,
            learning_rate=0.03,
            loss_function="RMSE",
            verbose=False,
            random_seed=42
        )


    else:

        raise ValueError(
            f"Unknown model: {name}"
        )


# ============================================================
# 5. REGRESSION MODEL NAMES
# ============================================================

regression_model_names = [

    "Linear Regression",
    "Ridge",
    "Lasso",
    "Polynomial Degree 2 + Ridge",
    "Polynomial Degree 3 + Ridge",
    "Extra Trees",
    "Random Forest",
    "Gradient Boosting",
    "CatBoost"
]


# ============================================================
# 6. COMPARE MODELS FOR ONE TARGET
# ============================================================

def evaluate_target(
    X,
    y,
    target_name
):

    results = []

    predictions = {}

    print("\n")
    print("=" * 70)
    print(
        f"{target_name.upper()} MODEL COMPARISON"
    )
    print("=" * 70)


    for model_name in regression_model_names:

        # IMPORTANT:
        # Create a completely fresh model
        model = make_regression_model(
            model_name
        )

        pred = cross_val_predict(
            model,
            X,
            y,
            cv=kf
        )

        predictions[
            model_name
        ] = pred


        rmse = np.sqrt(
            mean_squared_error(
                y,
                pred
            )
        )

        mae = mean_absolute_error(
            y,
            pred
        )

        r2 = r2_score(
            y,
            pred
        )


        results.append({

            "Model":
                model_name,

            "CV_RMSE":
                rmse,

            "CV_MAE":
                mae,

            "CV_R2":
                r2
        })


    results_df = (
        pd.DataFrame(results)
        .sort_values(
            "CV_RMSE"
        )
        .reset_index(drop=True)
    )


    print(
        results_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )


    best_name = (
        results_df.iloc[0]["Model"]
    )


    best_pred = predictions[
        best_name
    ]


    print("\n")
    print("-" * 70)
    print("BEST MODEL")
    print("-" * 70)

    print(
        "Selected model:",
        best_name
    )

    print(
        f"CV RMSE = "
        f"{results_df.iloc[0]['CV_RMSE']:.4f}"
    )

    print(
        f"CV MAE  = "
        f"{results_df.iloc[0]['CV_MAE']:.4f}"
    )

    print(
        f"CV R²   = "
        f"{results_df.iloc[0]['CV_R2']:.4f}"
    )


    return (
        best_name,
        results_df,
        best_pred
    )


# ============================================================
# 7. TARGETS
# ============================================================

targets = [

    "MeltPoolWidth_um",

    "MeltPoolDepth_um",

    "RelativeDensity_frac",

    "UTS_MPa"
]


# ============================================================
# 8. TRAIN FINAL MODELS
# ============================================================

final_models = {}

all_results = {}


for target in targets:

    y = df[target].values


    # --------------------------------------------------------
    # Compare models
    # --------------------------------------------------------

    (
        best_name,
        results_df,
        best_cv_pred
    ) = evaluate_target(
        X,
        y,
        target
    )


    all_results[
        target
    ] = results_df


    # --------------------------------------------------------
    # CRITICAL:
    # Create a BRAND NEW model
    # --------------------------------------------------------

    final_model = make_regression_model(
        best_name
    )


    # --------------------------------------------------------
    # Fit only this target
    # --------------------------------------------------------

    final_model.fit(
        X,
        y
    )


    # --------------------------------------------------------
    # Save in memory
    # --------------------------------------------------------

    final_models[
        target
    ] = final_model


# ============================================================
# 9. SAVE EACH MODEL
# ============================================================

joblib.dump(
    final_models["MeltPoolWidth_um"],
    "MeltPoolWidth_um.pkl"
)

joblib.dump(
    final_models["MeltPoolDepth_um"],
    "MeltPoolDepth_um.pkl"
)

joblib.dump(
    final_models["RelativeDensity_frac"],
    "RelativeDensity_frac.pkl"
)

joblib.dump(
    final_models["UTS_MPa"],
    "UTS_MPa.pkl"
)


# ============================================================
# 10. DEFECT CLASSIFIER
# ============================================================

print("\n")
print("=" * 70)
print("DEFECT CLASSIFICATION")
print("=" * 70)


y_cls = df[
    "DefectClass"
].astype(int).values


skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# CLASSIFICATION MODELS
# ============================================================

classification_models = {

    "Random Forest":
        RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),

    "Extra Trees":
        ExtraTreesClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            n_estimators=150,
            max_depth=3,
            learning_rate=0.03,
            random_state=42
        ),

    "CatBoost":
        CatBoostClassifier(
            iterations=300,
            depth=5,
            learning_rate=0.03,
            loss_function="MultiClass",
            verbose=False,
            random_seed=42
        )
}


classification_results = []

classification_predictions = {}


for name, model in classification_models.items():

    pred = cross_val_predict(
        model,
        X,
        y_cls,
        cv=skf
    )


    classification_predictions[
        name
    ] = pred


    accuracy = accuracy_score(
        y_cls,
        pred
    )

    f1 = f1_score(
        y_cls,
        pred,
        average="weighted"
    )


    classification_results.append({

        "Model":
            name,

        "CV_Accuracy":
            accuracy,

        "CV_Weighted_F1":
            f1
    })


classification_results_df = (
    pd.DataFrame(
        classification_results
    )
    .sort_values(
        "CV_Weighted_F1",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    classification_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 11. BEST CLASSIFIER
# ============================================================

best_classifier_name = (
    classification_results_df.iloc[0]["Model"]
)


best_classifier = (
    classification_models[
        best_classifier_name
    ]
)


best_classifier.fit(
    X,
    y_cls
)


joblib.dump(
    best_classifier,
    "defect_classifier.pkl"
)


best_classifier_pred = (
    classification_predictions[
        best_classifier_name
    ]
)


print("\n")
print("-" * 70)
print("BEST DEFECT CLASSIFIER")
print("-" * 70)

print(
    "Selected model:",
    best_classifier_name
)

print(
    f"CV Accuracy = "
    f"{classification_results_df.iloc[0]['CV_Accuracy']:.4f}"
)

print(
    f"CV Weighted F1 = "
    f"{classification_results_df.iloc[0]['CV_Weighted_F1']:.4f}"
)


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_cls,
        best_classifier_pred
    )
)


# ============================================================
# 12. SAVE MODEL INFORMATION
# ============================================================

model_info = {

    "features":
        features,

    "MeltPoolWidth_model":
        all_results[
            "MeltPoolWidth_um"
        ].iloc[0]["Model"],

    "MeltPoolDepth_model":
        all_results[
            "MeltPoolDepth_um"
        ].iloc[0]["Model"],

    "RelativeDensity_model":
        all_results[
            "RelativeDensity_frac"
        ].iloc[0]["Model"],

    "UTS_model":
        all_results[
            "UTS_MPa"
        ].iloc[0]["Model"],

    "DefectClassifier":
        best_classifier_name
}


joblib.dump(
    model_info,
    "final_model_info.pkl"
)


# ============================================================
# 13. FINAL SAVED-MODEL TEST
# ============================================================

print("\n")
print("=" * 70)
print("FINAL SAVED-MODEL TEST")
print("=" * 70)


test_X = pd.DataFrame([{

    "P_W": 270,

    "v_mm_per_s": 1500,

    "h_mm": 0.08,

    "t_mm": 0.04
}])


width_model = joblib.load(
    "MeltPoolWidth_um.pkl"
)

depth_model = joblib.load(
    "MeltPoolDepth_um.pkl"
)

density_model = joblib.load(
    "RelativeDensity_frac.pkl"
)

uts_model = joblib.load(
    "UTS_MPa.pkl"
)

defect_model = joblib.load(
    "defect_classifier.pkl"
)


# ============================================================
# PREDICTIONS
# ============================================================

width_prediction = float(
    width_model.predict(test_X)[0]
)

depth_prediction = float(
    depth_model.predict(test_X)[0]
)

density_prediction = float(
    density_model.predict(test_X)[0]
)

uts_prediction = float(
    uts_model.predict(test_X)[0]
)

defect_prediction = int(
    defect_model.predict(test_X)[0]
)


# Physical constraint
density_prediction = np.clip(
    density_prediction,
    0.0,
    1.0
)


print("\nTest condition:")

print(
    test_X.to_string(
        index=False
    )
)


print("\nPredictions:")

print(
    f"Melt Pool Width  = "
    f"{width_prediction:.3f} µm"
)

print(
    f"Melt Pool Depth  = "
    f"{depth_prediction:.3f} µm"
)

print(
    f"Relative Density = "
    f"{density_prediction:.6f}"
)

print(
    f"UTS              = "
    f"{uts_prediction:.3f} MPa"
)

print(
    f"Defect Class      = "
    f"{defect_prediction}"
)


# ============================================================
# 14. FINAL MODEL FILES
# ============================================================

print("\n")
print("=" * 70)
print("FILES SAVED")
print("=" * 70)

print(
    "MeltPoolWidth_um.pkl"
)

print(
    "MeltPoolDepth_um.pkl"
)

print(
    "RelativeDensity_frac.pkl"
)

print(
    "UTS_MPa.pkl"
)

print(
    "defect_classifier.pkl"
)

print(
    "feature_order.pkl"
)

print(
    "final_model_info.pkl"
)

print("\nTraining complete.")

DATASET
Shape: (500, 12)


MELTPOOLWIDTH_UM MODEL COMPARISON
                      Model  CV_RMSE  CV_MAE  CV_R2
                   CatBoost   6.3732  5.0455 0.9554
          Gradient Boosting   6.7046  5.3458 0.9506
              Random Forest   6.8447  5.4034 0.9486
                Extra Trees   6.9955  5.5468 0.9463
Polynomial Degree 3 + Ridge   7.1488  5.6115 0.9439
Polynomial Degree 2 + Ridge   8.1093  6.4580 0.9278
          Linear Regression   9.9162  8.0262 0.8920
                      Ridge   9.9391  7.9817 0.8915
                      Lasso  10.0053  7.9743 0.8901


----------------------------------------------------------------------
BEST MODEL
----------------------------------------------------------------------
Selected model: CatBoost
CV RMSE = 6.3732
CV MAE  = 5.0455
CV R²   = 0.9554


MELTPOOLDEPTH_UM MODEL COMPARISON
                      Model  CV_RMSE  CV_MAE  CV_R2
                   CatBoost  13.3576  9.7484 0.9743
          Gradient Boosting  13.8371 10.3591 0.9

C:\Users\satya\AppData\Local\Temp\ipykernel_27344\726628647.py:768: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  defect_prediction = int(


In [1]:
# ============================================================
# GROUPED REAL-DATA VALIDATION + 500-POINT JOINT SURROGATE
# ============================================================

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# Locate the data whether the notebook kernel starts at the workspace
# root or inside the 7th project folder.
DATA_DIR = Path.cwd()
if not (DATA_DIR / "hardness_experimental.csv").exists():
    DATA_DIR = DATA_DIR / "7th"

PROCESS_FEATURES = [
    "LaserPower_W",
    "ScanSpeed_mm_s",
    "HatchDistance_mm",
    "LayerThickness_mm",
]
MODEL_FEATURES = [
    "LaserPower_W",
    "ScanSpeed_mm_s",
    "HatchDistance_mm",
    "LayerThickness_mm",
]
TARGETS = ["UTS_MPa", "RelativeDensity_pct", "Hardness_HV"]

hardness_data = pd.read_csv(DATA_DIR / "hardness_experimental.csv")
uts_density_data = pd.read_csv(DATA_DIR / "uts_density_experimental.csv")
real_data = hardness_data.merge(
    uts_density_data,
    on="Sample",
    how="inner",
    validate="one_to_one",
)

assert len(real_data) == 20
assert set(real_data["Sample"]) == set(range(1, 21))
assert set(real_data.loc[real_data["Split"] == "validation", "Sample"]) == {
    4, 11, 16, 17, 18, 19, 20
}
assert np.isclose(
    real_data.loc[real_data["Sample"] == 18, "Parallel_1_HV"].iloc[0],
    320.0,
)

split_counts = real_data.groupby(PROCESS_FEATURES)["Split"].nunique()
assert split_counts.max() == 1, "A process condition has mixed train/validation rows."

aggregation = {
    target: (target, "mean")
    for target in TARGETS
}
aggregation.update({
    f"{target}_SD": (target, "std")
    for target in TARGETS
})
aggregation["Replicates"] = ("Sample", "count")
aggregation["Split"] = ("Split", "first")

condition_data = (
    real_data
    .groupby(PROCESS_FEATURES, as_index=False)
    .agg(**aggregation)
)

train_conditions = condition_data.loc[
    condition_data["Split"] == "train"
].reset_index(drop=True)
validation_conditions = condition_data.loc[
    condition_data["Split"] == "validation"
].reset_index(drop=True)

assert len(condition_data) == 15
assert len(train_conditions) == 13
assert len(validation_conditions) == 2

for feature in MODEL_FEATURES:
    assert validation_conditions[feature].between(
        train_conditions[feature].min(),
        train_conditions[feature].max(),
    ).all(), f"Validation {feature} is outside the training range."


def make_teacher_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=10.0)),
    ])


def generate_synthetic_points(source_conditions, count, seed):
    rng = np.random.default_rng(seed)
    bounds = {
        feature: (
            float(source_conditions[feature].min()),
            float(source_conditions[feature].max()),
        )
        for feature in MODEL_FEATURES
    }
    ved_values = (
        source_conditions["LaserPower_W"]
        / (
            source_conditions["ScanSpeed_mm_s"]
            * source_conditions["HatchDistance_mm"]
            * source_conditions["LayerThickness_mm"]
        )
    )
    ved_min = float(ved_values.min())
    ved_max = float(ved_values.max())
    batches = []
    accepted = 0

    for _ in range(20):
        pool_size = max(1000, (count - accepted) * 10)
        batch = pd.DataFrame({
            feature: (
                np.full(pool_size, bounds[feature][0])
                if bounds[feature][0] == bounds[feature][1]
                else rng.uniform(*bounds[feature], pool_size)
            )
            for feature in MODEL_FEATURES
        })
        batch["VED_J_mm3"] = batch["LaserPower_W"] / (
            batch["ScanSpeed_mm_s"]
            * batch["HatchDistance_mm"]
            * batch["LayerThickness_mm"]
        )
        batch = batch.loc[
            batch["VED_J_mm3"].between(ved_min, ved_max)
        ]
        if not batch.empty:
            batches.append(batch)
            accepted += len(batch)
        if accepted >= count:
            break

    if accepted < count:
        raise RuntimeError(
            f"Only generated {accepted} feasible points; requested {count}."
        )

    return pd.concat(batches, ignore_index=True).head(count)


def fit_teacher_students(source_conditions, seed):
    rng = np.random.default_rng(seed)
    synthetic_inputs = generate_synthetic_points(
        source_conditions,
        count=500,
        seed=seed,
    )
    synthetic_training = synthetic_inputs.copy()
    teacher_models = {}
    cv_metrics = []
    final_models = {}

    for target in TARGETS:
        features = source_conditions[MODEL_FEATURES]
        observed = source_conditions[target].to_numpy(dtype=float)
        cv_prediction = cross_val_predict(
            make_teacher_model(),
            features,
            observed,
            cv=LeaveOneOut(),
        )
        residuals = observed - cv_prediction
        centered_residuals = residuals - residuals.mean()

        teacher = make_teacher_model()
        teacher.fit(features, observed)
        teacher_models[target] = teacher

        synthetic_values = teacher.predict(synthetic_inputs[MODEL_FEATURES])
        synthetic_values += rng.choice(
            centered_residuals,
            size=len(synthetic_values),
            replace=True,
        )
        if target == "RelativeDensity_pct":
            synthetic_values = np.clip(synthetic_values, 0.0, 100.0)
        else:
            synthetic_values = np.maximum(synthetic_values, 0.0)
        synthetic_training[target] = synthetic_values

        cv_metrics.append({
            "Property": target,
            "LOOCV_RMSE": float(np.sqrt(mean_squared_error(observed, cv_prediction))),
            "LOOCV_MAE": float(mean_absolute_error(observed, cv_prediction)),
            "LOOCV_MeanError": float(np.mean(observed - cv_prediction)),
        })

        fit_features = pd.concat(
            [synthetic_inputs[MODEL_FEATURES], features],
            ignore_index=True,
        )
        fit_values = np.concatenate([synthetic_values, observed])
        fit_weights = np.concatenate([
            np.ones(len(synthetic_values)),
            np.full(len(observed), len(synthetic_values) / len(observed)),
        ])
        student = ExtraTreesRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42,
        )
        student.fit(fit_features, fit_values, sample_weight=fit_weights)
        final_models[target] = student

    return final_models, synthetic_training, pd.DataFrame(cv_metrics)


# Validation fold: both selected settings are excluded before the 500
# synthetic training points are generated.
validation_models, validation_synthetic_500, train_cv_metrics = fit_teacher_students(
    train_conditions,
    seed=42,
)

validation_rows = []
for _, condition in validation_conditions.iterrows():
    condition_input = pd.DataFrame([condition[MODEL_FEATURES].to_dict()])
    for target in TARGETS:
        prediction = float(validation_models[target].predict(condition_input)[0])
        validation_rows.append({
            **{feature: condition[feature] for feature in MODEL_FEATURES},
            "Property": target,
            "Actual_Mean": condition[target],
            "Actual_SD": condition[f"{target}_SD"],
            "Replicates": int(condition["Replicates"]),
            "Predicted": prediction,
            "Error": prediction - condition[target],
            "Absolute_Error": abs(prediction - condition[target]),
        })

validation_results = pd.DataFrame(validation_rows)
validation_results.to_csv(
    DATA_DIR / "joint_holdout_validation_results.csv",
    index=False,
)
train_cv_metrics.to_csv(
    DATA_DIR / "joint_train_loocv_metrics.csv",
    index=False,
)
validation_synthetic_500.to_csv(
    DATA_DIR / "joint_validation_fold_synthetic_500.csv",
    index=False,
)

print("Grouped real-data split:")
print(f"  Training conditions: {len(train_conditions)}")
print(f"  Held-out conditions: {len(validation_conditions)}")
print("  Held-out settings: 270/1500/0.08/0.04 and 280/1550/0.085/0.04")
print("\nTraining-only LOOCV metrics:")
display(train_cv_metrics.round(4))
print("\nReal held-out validation results:")
display(validation_results.round(4))


# Final fit after validation: retrain on all 15 real settings, create a
# fresh 500-point synthetic set, and save the joint forward models.
final_models, final_synthetic_500, final_loocv_metrics = fit_teacher_students(
    condition_data,
    seed=2026,
)
final_synthetic_500.to_csv(
    DATA_DIR / "joint_final_synthetic_training_500.csv",
    index=False,
)

model_dir = DATA_DIR / "models"
model_dir.mkdir(exist_ok=True)
for target, model in final_models.items():
    joblib.dump(model, model_dir / f"{target}_joint_500.pkl")
joblib.dump(MODEL_FEATURES, model_dir / "joint_500_feature_order.pkl")

# Rank a 500-point candidate sweep on the measured process domain.
candidate_inputs = final_synthetic_500[MODEL_FEATURES + ["VED_J_mm3"]].copy()
candidate_predictions = candidate_inputs.copy()
for target, model in final_models.items():
    candidate_predictions[target] = model.predict(
        candidate_inputs[MODEL_FEATURES]
    )

observed_ranges = condition_data[TARGETS].agg(["min", "max"])
normalized = pd.DataFrame(index=candidate_predictions.index)
for target in TARGETS:
    target_range = observed_ranges.loc["max", target] - observed_ranges.loc["min", target]
    normalized[target] = (
        candidate_predictions[target] - observed_ranges.loc["min", target]
    ) / target_range if target_range else 0.0
candidate_predictions["BalancedScore"] = normalized.mean(axis=1)

objective_values = candidate_predictions[TARGETS].to_numpy()
pareto_mask = np.ones(len(candidate_predictions), dtype=bool)
for index, value in enumerate(objective_values):
    dominates = np.all(objective_values >= value, axis=1) & np.any(
        objective_values > value,
        axis=1,
    )
    if dominates.any():
        pareto_mask[index] = False
candidate_predictions["Pareto"] = pareto_mask

pareto_pool = (
    candidate_predictions.loc[pareto_mask]
    .sort_values("BalancedScore", ascending=False)
    .head(30)
)
scaled_parameters = pareto_pool[MODEL_FEATURES].copy()
for feature in MODEL_FEATURES:
    span = condition_data[feature].max() - condition_data[feature].min()
    scaled_parameters[feature] = (
        (scaled_parameters[feature] - condition_data[feature].min()) / span
        if span else 0.0
    )

selected_indices = [pareto_pool.index[0]]
while len(selected_indices) < min(6, len(pareto_pool)):
    remaining = [index for index in pareto_pool.index if index not in selected_indices]
    remaining_values = scaled_parameters.loc[remaining].to_numpy()
    selected_values = scaled_parameters.loc[selected_indices].to_numpy()
    distances = np.linalg.norm(
        remaining_values[:, None, :] - selected_values[None, :, :],
        axis=2,
    ).min(axis=1)
    selected_indices.append(remaining[int(np.argmax(distances))])

top_candidates = (
    candidate_predictions.loc[selected_indices]
    .sort_values("BalancedScore", ascending=False)
    .reset_index(drop=True)
)
candidate_predictions.to_csv(
    DATA_DIR / "joint_500_candidate_predictions.csv",
    index=False,
)
top_candidates.to_csv(
    DATA_DIR / "joint_top_6_candidates.csv",
    index=False,
)

print("\nTop diverse Pareto candidates (final models trained on all 15 settings):")
display(top_candidates.round(4))

Grouped real-data split:
  Training conditions: 13
  Held-out conditions: 2
  Held-out settings: 270/1500/0.08/0.04 and 280/1550/0.085/0.04

Training-only LOOCV metrics:


,Property,LOOCV_RMSE,LOOCV_MAE,LOOCV_MeanError
0,UTS_MPa,38.7408,27.0177,0.4987
1,RelativeDensity_pct,0.4754,0.3479,-0.0061
2,Hardness_HV,7.3892,6.2419,0.1019



Real held-out validation results:


,LaserPower_W,ScanSpeed_mm_s,HatchDistance_mm,LayerThickness_mm,Property,Actual_Mean,Actual_SD,Replicates,Predicted,Error,Absolute_Error
0,270,1500,0.080,0.04,UTS_MPa,890.0000,21.2038,6,902.4771,12.4771,12.4771
1,270,1500,0.080,0.04,RelativeDensity_pct,99.6167,0.3061,6,99.4497,-0.1670,0.1670
2,270,1500,0.080,0.04,Hardness_HV,324.8972,3.5454,6,328.3128,3.4156,3.4156
3,280,1550,0.085,0.04,UTS_MPa,923.0000,NaN,1,875.1346,-47.8654,47.8654
4,280,1550,0.085,0.04,RelativeDensity_pct,99.9500,NaN,1,99.5914,-0.3586,0.3586
5,280,1550,0.085,0.04,Hardness_HV,323.3750,NaN,1,322.4700,-0.9050,0.9050



Top diverse Pareto candidates (final models trained on all 15 settings):


,LaserPower_W,ScanSpeed_mm_s,HatchDistance_mm,LayerThickness_mm,VED_J_mm3,UTS_MPa,RelativeDensity_pct,Hardness_HV,BalancedScore,Pareto
0,269.6593,1402.1685,0.0801,0.04,60.0172,906.1773,99.7898,331.7025,0.8409,True
1,273.6568,1587.1985,0.0709,0.04,60.7738,907.6654,99.3415,328.3091,0.7025,True
2,260.8272,1510.8074,0.0732,0.04,58.9895,908.4007,99.2852,327.7494,0.6842,True
3,265.8141,1585.9208,0.0785,0.04,53.3981,907.0519,99.3468,327.1189,0.6823,True
4,252.1892,1590.2580,0.0728,0.04,54.4741,932.2374,98.9955,327.5906,0.6808,True
5,261.8539,1515.5668,0.0825,0.04,52.3563,906.3724,99.4231,325.7680,0.6728,True


In [2]:
# Save models trained without either held-out setting so V5 can perform
# retrospective inverse-recovery checks without validation leakage.
validation_model_dir = DATA_DIR / "models" / "joint_validation_fold"
validation_model_dir.mkdir(parents=True, exist_ok=True)
for target, model in validation_models.items():
    joblib.dump(
        model,
        validation_model_dir / f"{target}_joint_validation.pkl",
    )
joblib.dump(
    MODEL_FEATURES,
    validation_model_dir / "joint_validation_feature_order.pkl",
)
validation_conditions.to_csv(
    DATA_DIR / "joint_validation_conditions.csv",
    index=False,
)

print(f"Saved validation-only models to: {validation_model_dir}")
print("Saved measured grouped targets to joint_validation_conditions.csv")

Saved validation-only models to: d:\Sem 7\UG\7th\models\joint_validation_fold
Saved measured grouped targets to joint_validation_conditions.csv


In [3]:
# Add interpretable percentage-error metrics to real-data validation reports.
actual_values = validation_results["Actual_Mean"].to_numpy(dtype=float)
error_values = validation_results["Error"].to_numpy(dtype=float)
validation_results["ErrorPercent"] = np.divide(
    100.0 * error_values,
    np.abs(actual_values),
    out=np.full_like(error_values, np.nan),
    where=np.abs(actual_values) > np.finfo(float).eps,
)
validation_results["AbsoluteErrorPercent"] = validation_results["ErrorPercent"].abs()
validation_results.to_csv(
    DATA_DIR / "joint_holdout_validation_results.csv",
    index=False,
)

percentage_metrics = []
for target in TARGETS:
    actual = train_conditions[target].to_numpy(dtype=float)
    cv_prediction = cross_val_predict(
        make_teacher_model(),
        train_conditions[MODEL_FEATURES],
        actual,
        cv=LeaveOneOut(),
    )
    signed_percent_error = np.divide(
        100.0 * (cv_prediction - actual),
        np.abs(actual),
        out=np.full_like(actual, np.nan),
        where=np.abs(actual) > np.finfo(float).eps,
    )
    percentage_metrics.append({
        "Property": target,
        "LOOCV_MAPE_Percent": float(np.nanmean(np.abs(signed_percent_error))),
        "LOOCV_MeanSignedPercentError": float(np.nanmean(signed_percent_error)),
    })

percentage_metrics = pd.DataFrame(percentage_metrics)
train_cv_metrics = (
    train_cv_metrics
    .drop(
        columns=["LOOCV_MAPE_Percent", "LOOCV_MeanSignedPercentError"],
        errors="ignore",
    )
    .merge(percentage_metrics, on="Property", how="left", validate="one_to_one")
)
train_cv_metrics.to_csv(
    DATA_DIR / "joint_train_loocv_metrics.csv",
    index=False,
)

print("Forward holdout results with percentage errors:")
display(validation_results.round(4))
print("Training-only LOOCV metrics with percentage errors:")
display(train_cv_metrics.round(4))

Forward holdout results with percentage errors:


,LaserPower_W,ScanSpeed_mm_s,HatchDistance_mm,LayerThickness_mm,Property,Actual_Mean,Actual_SD,Replicates,Predicted,Error,Absolute_Error,ErrorPercent,AbsoluteErrorPercent
0,270,1500,0.080,0.04,UTS_MPa,890.0000,21.2038,6,902.4771,12.4771,12.4771,1.4019,1.4019
1,270,1500,0.080,0.04,RelativeDensity_pct,99.6167,0.3061,6,99.4497,-0.1670,0.1670,-0.1676,0.1676
2,270,1500,0.080,0.04,Hardness_HV,324.8972,3.5454,6,328.3128,3.4156,3.4156,1.0513,1.0513
3,280,1550,0.085,0.04,UTS_MPa,923.0000,NaN,1,875.1346,-47.8654,47.8654,-5.1859,5.1859
4,280,1550,0.085,0.04,RelativeDensity_pct,99.9500,NaN,1,99.5914,-0.3586,0.3586,-0.3588,0.3588
5,280,1550,0.085,0.04,Hardness_HV,323.3750,NaN,1,322.4700,-0.9050,0.9050,-0.2799,0.2799


Training-only LOOCV metrics with percentage errors:


,Property,LOOCV_RMSE,LOOCV_MAE,LOOCV_MeanError,LOOCV_MAPE_Percent,LOOCV_MeanSignedPercentError
0,UTS_MPa,38.7408,27.0177,0.4987,3.1053,0.1128
1,RelativeDensity_pct,0.4754,0.3479,-0.0061,0.3508,0.0082
2,Hardness_HV,7.3892,6.2419,0.1019,1.9204,0.0118


In [4]:
# Add scale-aware and standard regression metrics from real-data predictions.
from sklearn.metrics import r2_score

cv_metric_rows = []
for target in TARGETS:
    actual = train_conditions[target].to_numpy(dtype=float)
    cv_prediction = cross_val_predict(
        make_teacher_model(),
        train_conditions[MODEL_FEATURES],
        actual,
        cv=LeaveOneOut(),
    )
    error = cv_prediction - actual
    signed_percent_error = np.divide(
        100.0 * error,
        np.abs(actual),
        out=np.full_like(error, np.nan),
        where=np.abs(actual) > np.finfo(float).eps,
    )
    cv_metric_rows.append({
        "Property": target,
        "N_Conditions": len(actual),
        "MSE": float(mean_squared_error(actual, cv_prediction)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, cv_prediction))),
        "MAE": float(mean_absolute_error(actual, cv_prediction)),
        "R2": float(r2_score(actual, cv_prediction)),
        "MAPE_Percent": float(np.nanmean(np.abs(signed_percent_error))),
        "MeanSignedPercentError": float(np.nanmean(signed_percent_error)),
    })

cv_metrics_report = pd.DataFrame(cv_metric_rows)
cv_metrics_report.to_csv(
    DATA_DIR / "joint_train_loocv_metrics.csv",
    index=False,
)

holdout_metric_rows = []
for target, group in validation_results.groupby("Property"):
    actual = group["Actual_Mean"].to_numpy(dtype=float)
    predicted = group["Predicted"].to_numpy(dtype=float)
    errors = predicted - actual
    signed_percent_error = np.divide(
        100.0 * errors,
        np.abs(actual),
        out=np.full_like(errors, np.nan),
        where=np.abs(actual) > np.finfo(float).eps,
    )
    holdout_metric_rows.append({
        "Property": target,
        "N_Independent_Conditions": len(group),
        "MSE": float(mean_squared_error(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "MAE": float(mean_absolute_error(actual, predicted)),
        "R2": float(r2_score(actual, predicted)) if len(group) >= 2 else np.nan,
        "MAPE_Percent": float(np.nanmean(np.abs(signed_percent_error))),
        "R2_Caution": "Only two held-out conditions; do not interpret as stable validation R2",
    })

holdout_metrics_report = pd.DataFrame(holdout_metric_rows)
holdout_metrics_report.to_csv(
    DATA_DIR / "joint_holdout_summary_metrics.csv",
    index=False,
)

print("Training-only leave-one-condition-out metrics:")
display(cv_metrics_report.round(4))
print("Two-condition holdout summary; R2 is exploratory only:")
display(holdout_metrics_report.round(4))

Training-only leave-one-condition-out metrics:


,Property,N_Conditions,MSE,RMSE,MAE,R2,MAPE_Percent,MeanSignedPercentError
0,UTS_MPa,13,1500.8517,38.7408,27.0177,-0.3047,3.1053,0.1128
1,RelativeDensity_pct,13,0.2260,0.4754,0.3479,-0.0616,0.3508,0.0082
2,Hardness_HV,13,54.6007,7.3892,6.2419,-0.3467,1.9204,0.0118


Two-condition holdout summary; R2 is exploratory only:


,Property,N_Independent_Conditions,MSE,RMSE,MAE,R2,MAPE_Percent,R2_Caution
0,Hardness_HV,2,6.2426,2.4985,2.1603,-9.7764,0.6656,Only two held-out conditions; do not interpret...
1,RelativeDensity_pct,2,0.0782,0.2797,0.2628,-1.8166,0.2632,Only two held-out conditions; do not interpret...
2,UTS_MPa,2,1223.3893,34.9770,30.1713,-3.4936,3.2939,Only two held-out conditions; do not interpret...
